# ROW supplement: fixed S = 2 / 3 / 4 identity slots x {Transformer AR, Looped AR} = 6 runs

Standalone supplement to notebook 18. It does NOT touch notebook 18's 26-job queue or its results.

| condition | S | train k | held-out test | test records / model |
|---|---|---|---|---|
| identity_s2_train02_row | 2 | 0,2 | A/B/C at both slots | 16,000 |
| identity_s3_train023_row | 3 | 0,2,3 | A/B/C at all 3 slots | 46,000 |
| identity_s4_train0234_row | 4 | 0,2,3,4 | A/B/C at all 4 slots | 130,000 |

**Format.** Real `<ROW>` tokens between the rows of the input AND output grids, visible `<identity>` fillers, and
learned positions. Sequences are 89 / 90 / 91 tokens.

**Same machinery as notebook 18.** Serializer, vocabulary, model config, the common fresh initialization, the
42-target loss, digit-CE selection, and free-running unrepaired scoring.

**Pairing.**
* The logical stream equals the main S5 / variable stream at the same support.
* Placements and test manifests equal notebook 16's S=2/3/4 no-ROW runs.

**Comparisons.** Comparisons with notebook 18 and notebook 16 are read-only, and only use completed artifacts. Missing
ones show as pending.

**Send back `AR_Row_Slots234_Review.ipynb`** (printed and downloaded by the last cell). One seed: descriptive only.


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION ------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")      # "full" | "smoke" (tiny CPU check, separate root, NOT results)
SEED = 42
SUPPLEMENT_CONDITIONS = ["identity_s2_train02_row", "identity_s3_train023_row", "identity_s4_train0234_row"]
MODEL_IDS = ["transformer_ar", "looped_transformer_ar"]
RUN_STAGES = ["train", "evaluate", "analyze"]
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
DTYPE = "auto"
EVAL_CHUNK_GRIDS = 100
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import json, pathlib
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import rowsep as RS, rowsep_reports as RR, rowsep_slots as SP
from decompositionality.config import make_experiment_config
JOBS = SP.jobs(SUPPLEMENT_CONDITIONS, MODEL_IDS)
OUTPUT_ROOT = RR.family_root(RESULTS_ROOT, MODE, SP.SUBDIR)
MAIN_ROOT = RR.family_root(RESULTS_ROOT, MODE)          # notebook-18 root: READ-ONLY
_c = make_experiment_config(JOBS[0][0], MODE, str(pathlib.Path(RESULTS_ROOT) / SP.SUBDIR), MODEL_IDS, seed=SEED)
print("resolved recipe:", _c.train.semantic_dict())
print(f"{len(JOBS)} supplement jobs:", JOBS)
print("output ->", OUTPUT_ROOT, "| main suite (read-only) ->", MAIN_ROOT)
if MODE == "smoke":
    print("*** SMOKE MODE - NOT results ***")


In [ ]:
# --- Preflight (stop on failure): layouts, counts, stream / placement / init matching -------------------------------
AUD = OUTPUT_ROOT / "audits" / f"seed_{SEED}"; AUD.mkdir(parents=True, exist_ok=True)
pf, ini = SP.preflight(SEED, main_root=MAIN_ROOT)
lay = RR.layout_audit()
for name, df in (("preflight", pf), ("initialization", ini), ("layout", lay)):
    df.to_csv(AUD / f"{name}.csv", index=False); print(f"\n[{name}]"); display(df)


In [ ]:
# --- Sequential six-job driver (resumable; one model on the GPU at a time) ------------------------------------------
status = RR.run_queue(RESULTS_ROOT, MODE, SEED, JOBS, RUN_STAGES, repo_dir=str(REPO_DIR), chunk_grids=EVAL_CHUNK_GRIDS,
                      device=DEVICE, dtype=DTYPE, subdir=SP.SUBDIR, universe=SP.jobs())
display(status)


In [ ]:
# --- Report + read-only comparisons + review export (rerunnable alone; never trains) ---------------------------
rp = SP.report(RESULTS_ROOT, MODE, SEED, JOBS)
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp))
